# RNN Model Univariate

In this section we implement the RNN Model (Recurrent Neural Network) using the **TimeSeriesDataset** approach with one-hot encoding.

The RNN (Recurrent Neural Network) Forecaster is a vanilla RNN model designed for time series forecasting. It uses one-hot encoding to identify individual series (1502 unique series), processing one series at a time. It processes sequential data using recurrent connections that maintain a hidden state across time steps, allowing it to capture temporal dependencies.

**Layer Breakdown:**

- **RNN Layers:** 2 stacked RNN layers with Tanh activation
- **Hidden Size:** 64 units per layer
- **Dropout:** Applied between RNN layers (if >1 layer) and before final output
- **Output Layer:** Single fully connected layer producing 1-step forecast

**Advantages**

- **Simplicity:** Simpler architecture than LSTM/GRU (only hidden state, no cell state)
- **Speed:** Faster training due to fewer parameters
- **Sequential Processing:** Captures temporal dependencies in time series data

**Limitations**

- **Vanishing Gradients:** May struggle with long-term dependencies (sequences >10-20 steps)
- **Limited Memory:** No gating mechanisms to control information flow like LSTM/GRU


In [1]:
import torch
import torch.nn as nn

## Model

In [2]:
class RNNForecaster(nn.Module):
    """
    Vanilla RNN model for MULTIVARIATE time series forecasting.
    Architecture: RNN -> Dropout -> RNN -> Dropout -> Fully Connected
    Takes multiple input features at each timestep.
    
    Simpler than LSTM - no cell state, only hidden state.
    Faster training but may struggle with long-term dependencies.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: RNN hidden dimension
            num_layers: Number of RNN layers
            dropout: Dropout rate
        """
        super(RNNForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        
        # RNN layers (using Tanh activation by default)
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            nonlinearity='tanh'  # Can also use 'relu'
        )
        
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # RNN forward pass
        # rnn_out: (batch_size, seq_length, hidden_size)
        # h_n: (num_layers, batch_size, hidden_size)
        rnn_out, h_n = self.rnn(x)
        
        # Take the output from the last time step
        last_output = rnn_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Apply dropout
        out = self.dropout(last_output)
        
        # Fully connected layer
        out = self.fc(out)  # Shape: (batch_size, 1)
        
        return out

### Model Results without Exogenous Features

### Model Results with Exogenous Features
